In [2]:
# duckdb_schema_export.py
# Produces Markdown with CREATE TABLE DDL and small samples for each (non-system) table.
# Usage:
#   python duckdb_schema_export.py /path/to/db.duckdb > duckdb_schema.md
#
# Or import and call: export_duckdb_schema_markdown(db_path="...")

import sys
import duckdb
import pandas as pd
from typing import Optional, Iterable, Dict, List, Tuple

SYSTEM_SCHEMAS = {"information_schema", "pg_catalog"}

def qident(name: str) -> str:
    """Quote an SQL identifier if needed."""
    if name is None:
        return ""
    # simple, robust quoting
    s = name.replace('"', '""')
    return f'"{s}"'

def qualified(schema: str, table: str) -> str:
    return f"{qident(schema)}.{qident(table)}" if schema else qident(table)

def get_objects(con, include_views: bool = True) -> List[Tuple[str, str, str]]:
    # table_type: BASE TABLE | VIEW | LOCAL TEMPORARY
    types = ("'BASE TABLE'", "'VIEW'") if include_views else ("'BASE TABLE'",)
    q = f"""
      SELECT table_schema, table_name, table_type
      FROM information_schema.tables
      WHERE table_schema NOT IN ({",".join(["?"]*len(SYSTEM_SCHEMAS))})
        AND table_type IN ({",".join(types)})
      ORDER BY table_schema, table_name
    """
    return con.execute(q, list(SYSTEM_SCHEMAS)).fetchall()

def get_columns(con, schema: str, table: str):
    q = """
      SELECT column_name, data_type, is_nullable, column_default, ordinal_position
      FROM information_schema.columns
      WHERE table_schema = ? AND table_name = ?
      ORDER BY ordinal_position
    """
    return con.execute(q, [schema, table]).fetchall()

def get_primary_key(con, schema: str, table: str) -> List[str]:
    q = """
      SELECT kcu.column_name
      FROM information_schema.table_constraints tc
      JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema   = kcu.table_schema
       AND tc.table_name     = kcu.table_name
      WHERE tc.constraint_type = 'PRIMARY KEY'
        AND tc.table_schema = ? AND tc.table_name = ?
      ORDER BY kcu.ordinal_position
    """
    rows = con.execute(q, [schema, table]).fetchall()
    return [r[0] for r in rows]

def get_foreign_keys(con, schema: str, table: str):
    q = """
      SELECT kcu.column_name,
             ccu.table_schema  AS foreign_table_schema,
             ccu.table_name    AS foreign_table_name,
             ccu.column_name   AS foreign_column_name
      FROM information_schema.table_constraints tc
      JOIN information_schema.key_column_usage kcu
        ON tc.constraint_name = kcu.constraint_name
       AND tc.table_schema   = kcu.table_schema
       AND tc.table_name     = kcu.table_name
      JOIN information_schema.constraint_column_usage ccu
        ON tc.constraint_name = ccu.constraint_name
       AND tc.table_schema   = ccu.table_schema
      WHERE tc.constraint_type = 'FOREIGN KEY'
        AND tc.table_schema = ? AND tc.table_name = ?
      ORDER BY kcu.ordinal_position
    """
    return con.execute(q, [schema, table]).fetchall()

def render_create_table(schema: str, table: str, cols, pks, fks) -> str:
    lines = []
    for name, dtype, is_null, default, _ord in cols:
        col = f"  {qident(name)} {dtype}"
        if is_null == "NO":
            col += " NOT NULL"
        if default not in (None, ""):
            col += f" DEFAULT {default}"
        lines.append(col)
    if pks:
        lines.append(f"  , PRIMARY KEY ({', '.join(qident(c) for c in pks)})")
    for col, f_schema, f_table, f_col in fks:
        lines.append(
            "  , FOREIGN KEY ({col}) REFERENCES {ref}({refcol})".format(
                col=qident(col),
                ref=qualified(f_schema, f_table),
                refcol=qident(f_col),
            )
        )
    ddl = "CREATE TABLE {qt} (\n{cols}\n);".format(
        qt=qualified(schema, table), cols=",\n".join(lines)
    )
    return ddl

def sample_csv(con, schema: str, table: str, limit: int = 5) -> str:
    try:
        df = con.execute(f"SELECT * FROM {qualified(schema, table)} LIMIT {limit}").df()
        if df.empty:
            return ""
        # CSV is broadly LLM-friendly and preserves types textually.
        return df.to_csv(index=False)
    except Exception as e:
        return f"# Unable to fetch sample rows: {e}"

def export_duckdb_schema_markdown(
    db_path: Optional[str] = None,
    con: Optional[duckdb.DuckDBPyConnection] = None,
    include_views: bool = True,
    sample_rows: int = 5,
) -> str:
    close_at_end = False
    if con is None:
        if db_path is None:
            raise ValueError("Provide either db_path or an existing DuckDB connection.")
        con = duckdb.connect(db_path)
        close_at_end = True

    out_lines: List[str] = []
    objs = get_objects(con, include_views=include_views)
    if not objs:
        return "_No user tables/views found._"

    for schema, table, ttype in objs:
        cols = get_columns(con, schema, table)
        pks  = get_primary_key(con, schema, table)
        fks  = get_foreign_keys(con, schema, table)

        out_lines.append(f"## {schema}.{table} ({ttype})")
        ddl = render_create_table(schema, table, cols, pks, fks)
        out_lines.append("```sql")
        out_lines.append(ddl)
        out_lines.append("```")

        csv_text = sample_csv(con, schema, table, limit=sample_rows)
        if csv_text:
            out_lines.append("<details><summary>sample rows</summary>\n\n```csv")
            out_lines.append(csv_text.rstrip())
            out_lines.append("```\n</details>")

        out_lines.append("")  # spacing

    if close_at_end:
        con.close()

    return "\n".join(out_lines)

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python duckdb_schema_export.py ../data/mydb2024-25.duckdb", file=sys.stderr)
        sys.exit(2)
    db = "../data/mydb2024-25.duckdb"
    # db = "../data/hbl.duckdb"
    md = export_duckdb_schema_markdown(db_path=db, include_views=True, sample_rows=5)
    print(md)
    # to md file:
    with open("duckdb_schema.md", "w", encoding="utf-8") as f:
        f.write(md)

    # db = "../data/mydb2024-25.duckdb"
    db = "../data/hbl.duckdb"
    md = export_duckdb_schema_markdown(db_path=db, include_views=True, sample_rows=5)
    print(md)
    # to md file:
    with open("duckdb_schema2.md", "w", encoding="utf-8") as f:
        f.write(md)

## main.fixtures (BASE TABLE)
```sql
CREATE TABLE "main"."fixtures" (
  "fixtureId" VARCHAR,
  "seasonId" VARCHAR,
  "fixtureNumber" BIGINT,
  "nameLocal" VARCHAR,
  "nameLatin" INTEGER,
  "startTimeLocal" VARCHAR,
  "startTimeUTC" VARCHAR,
  "roundNumber" VARCHAR,
  "externalId" VARCHAR,
  "competitors" STRUCT(entityId VARCHAR, isHome BOOLEAN, draw BOOLEAN, resultPlace INTEGER, score VARCHAR, nameFullLocal VARCHAR)[],
  "entityId_home" VARCHAR,
  "entityId_away" VARCHAR,
  "name_team_home" VARCHAR,
  "name_team_away" VARCHAR,
  "score_home" VARCHAR,
  "score_away" VARCHAR,
  "resultPlace_home" BIGINT,
  "resultPlace_away" BIGINT,
  "session_id" BIGINT
);
```
<details><summary>sample rows</summary>

```csv
fixtureId,seasonId,fixtureNumber,nameLocal,nameLatin,startTimeLocal,startTimeUTC,roundNumber,externalId,competitors,entityId_home,entityId_away,name_team_home,name_team_away,score_home,score_away,resultPlace_home,resultPlace_away,session_id
00c08679-4374-11ef-80bd-73cf0bc66b45,cabcf5

In [ ]:
# con = duckdb.connect("../data/hbl.duckdb")
# # list all tables except the one to keep
# tables = con.execute("""
#     SELECT table_name
#     FROM information_schema.tables
#     WHERE table_schema='main'
#       AND table_name <> 'kinexon_positions'
# """).fetchdf()['table_name'].tolist()

# # drop them
# for t in tables:
#     con.execute(f'DROP TABLE IF EXISTS "{t}";')
# con.close()

In [ ]:
import duckdb

# Connect to the destination database
con = duckdb.connect('../data/hbl.duckdb')

# Attach the source database
con.execute("ATTACH '../data/mydb2024-25.duckdb' AS source_db")

# 1. Create table if it doesn't exist (populates it if created)
con.execute("CREATE TABLE IF NOT EXISTS kinexon_positions AS SELECT * FROM source_db.kinexon_positions")

# 2. Insert only new rows (avoiding duplicates)
# Using EXCEPT to filter out rows that already exist in the destination
con.execute("""
    INSERT INTO kinexon_positions
    SELECT * FROM source_db.kinexon_positions
    EXCEPT
    SELECT * FROM kinexon_positions
""")

# Verify the transfer
print("Tables in hbl.duckdb:")
print(con.execute("SHOW TABLES").df())
print(f"Total rows in kinexon_positions: {con.execute('SELECT COUNT(*) FROM kinexon_positions').fetchone()[0]}")

# Detach and close
con.execute("DETACH source_db")
con.close()